In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df_main = pd.read_csv("data/Cassava_Flood_Data_Coded - Sheet1.csv")
df_equip = pd.read_csv("data/Cassava_Flood_Data_Coded - Equipment used.csv")
df_coping = pd.read_csv("data/Cassava_Flood_Data_Coded - Coping and adaptation strategies.csv")
df_interv = pd.read_csv("data/Cassava_Flood_Data_Coded - Effectiveness of flood-management interventions.csv")


In [2]:
# Clean column names
df_main.columns = df_main.columns.str.strip()
df_equip.columns = df_equip.columns.str.strip()
df_coping.columns = df_coping.columns.str.strip()
df_interv.columns = df_interv.columns.str.strip()


# Helper function to safely fetch and coerce columns to numeric
def get_numeric_col(df, possible_names):
  for name in possible_names:
    if name in df.columns:
      return pd.to_numeric(df[name], errors="coerce").fillna(0)
  return pd.Series(0, index=df.index)

In [3]:
# 2. DATA PREPARATION & VARIABLE CONSTRUCTION (Objective i & ii)
total_harvest = get_numeric_col(
    df_main, ["Total_Harvest_Kg", "Total Harvest Kg"]
)
farm_size_ha = get_numeric_col(
    df_main, ["Farm_Size_in_Hectares", "Farm Size in Hectares"]
)
df_main["Yield_Kg_Ha"] = total_harvest / farm_size_ha.replace(0, np.nan)

price_per_kg = get_numeric_col(
    df_main, ["Price_Per_Kg_Naira", "Price Per Kg Naira"]
)
df_main["TR"] = total_harvest * price_per_kg

# Total Variable Cost (TVC)
hired_labor = get_numeric_col(
    df_main, ["Hired_Labor_Cost_Naira", "Hired Labor Cost Naira"]
)
family_labor = get_numeric_col(
    df_main, ["Family_Labour_Cost_Naira", "Family Labour Cost Naira"]
)
input_cost = get_numeric_col(
    df_main, ["Input_Cost_Naira", "Input Cost Naira"]
)
land_rent = get_numeric_col(df_main, ["Land_Rent_Naira", "Land Rent Naira"])
transport_cost = get_numeric_col(
    df_main, ["Transport_Cost_Naira", "Transport Cost Naira"]
)

df_main["TVC"] = (
    hired_labor + family_labor + input_cost + land_rent + transport_cost
)

# Total Fixed Cost (TFC) from Equipment depreciation
hoe_dep = get_numeric_col(
    df_equip, ["Annual depreciation(hoe)", "Annual depreciation (hoe)"]
)
cutlass_dep = get_numeric_col(
    df_equip, ["Annual depreciation(cutlass)", "Annual depreciation (cutlass)"]
)
others_dep = get_numeric_col(
    df_equip, ["Annual depreciation(others)", "Annual depreciation (others)"]
)
df_equip["TFC_Equipment"] = hoe_dep + cutlass_dep + others_dep

df_main = df_main.merge(
    df_equip[["Respondent_ID", "TFC_Equipment"]],
    on="Respondent_ID",
    how="left",
)
df_main["TFC"] = df_main["TFC_Equipment"].fillna(0)

# Profitability Metrics: TC, GM, NFI
df_main["TC"] = df_main["TVC"] + df_main["TFC"]
df_main["GM"] = df_main["TR"] - df_main["TVC"]
df_main["NFI"] = df_main["TR"] - df_main["TC"]

In [4]:
# 3. DESCRIPTIVE STATISTICS
continuous_vars = [
    "Age",
    "Household_Size",
    "Farm_Size_in_Hectares",
    "Yield_Kg_Ha",
    "TR",
    "GM",
    "NFI",
]
print("--- DESCRIPTIVE STATISTICS (Mean & Standard Deviation) ---")
desc_stats = df_main.groupby("Flood_Exposure")[continuous_vars].agg(
    ["mean", "std"]
)
print(desc_stats)

--- DESCRIPTIVE STATISTICS (Mean & Standard Deviation) ---
                   Age            Household_Size            \
                  mean        std           mean       std   
Flood_Exposure                                               
1               48.125  11.743383           2.30  0.939176   
2               45.650   9.973169           2.35  0.769615   

               Farm_Size_in_Hectares             Yield_Kg_Ha              \
                                mean       std          mean         std   
Flood_Exposure                                                             
1                            0.26225  0.157472  14531.039587  4772.02930   
2                            0.33575  0.166177   7882.511102  3359.64866   

                        TR                         GM                 \
                      mean            std        mean            std   
Flood_Exposure                                                         
1               651336.875  37655

In [5]:
# 4. HYPOTHESIS TESTING (Independent-Samples t-tests: Group 1 vs Group 2)
print("\n--- INDEPENDENT-SAMPLES T-TESTS (Affected vs Free) ---")
test_variables = ["Yield_Kg_Ha", "TR", "GM", "NFI"]

for var in test_variables:
  group_1 = df_main[df_main["Flood_Exposure"] == 1][var].dropna()
  group_2 = df_main[df_main["Flood_Exposure"] == 2][var].dropna()

  if len(group_1) > 1 and len(group_2) > 1:
    t_stat, p_value = stats.ttest_ind(group_1, group_2, equal_var=False)
    print(
        f"Variable: {var:12} | t-statistic: {t_stat:8.4f} | p-value:"
        f" {p_value:8.4f}"
    )
  else:
    print(f"Variable: {var:12} | Insufficient data for t-test")


--- INDEPENDENT-SAMPLES T-TESTS (Affected vs Free) ---
Variable: Yield_Kg_Ha  | t-statistic:   7.2050 | p-value:   0.0000
Variable: TR           | t-statistic:   1.8277 | p-value:   0.0714
Variable: GM           | t-statistic:   1.4819 | p-value:   0.1424
Variable: NFI          | t-statistic:   1.4738 | p-value:   0.1446


In [6]:
# 5. COPING & ADAPTATION STRATEGIES (Chi-Square Test - Objective iii)
print("\n--- COPING STRATEGIES CHI-SQUARE ANALYSIS ---")
df_coping_merged = df_coping.merge(
    df_main[["Respondent_ID", "Flood_Exposure"]], on="Respondent_ID", how="left"
)

coping_columns = [
    "Early harvesting",
    "Seeking other sources of income(non-farm activities)",
    "Reducing household expenditure",
    "Borrowing money",
    "Getting help from relatives or communiity members",
]

for col in coping_columns:
  if col in df_coping_merged.columns:
    df_coping_merged[col + "_bin"] = (
        df_coping_merged[col].str.strip().str.lower() == "yes"
    ).astype(int)
    contingency = pd.crosstab(
        df_coping_merged["Flood_Exposure"], df_coping_merged[col + "_bin"]
    )
    if contingency.shape == (2, 2):
      chi2, p, dof, _ = stats.chi2_contingency(contingency)
      print(f"Strategy ({col[:30]}...) | Chi2: {chi2:.4f} | p-value: {p:.4f}")


--- COPING STRATEGIES CHI-SQUARE ANALYSIS ---


Strategy (Early harvesting...) | Chi2: 12.8080 | p-value: 0.0003
Strategy (Seeking other sources of incom...) | Chi2: 0.8427 | p-value: 0.3586
Strategy (Reducing household expenditure...) | Chi2: 5.0031 | p-value: 0.0253
Strategy (Borrowing money...) | Chi2: 0.2164 | p-value: 0.6418
Strategy (Getting help from relatives or...) | Chi2: 1.2531 | p-value: 0.2630


In [7]:
# 6. POLICY & INTERVENTION EFFECTIVENESS SCORING (Objective iv)
print("\n--- INTERVENTION EFFECTIVENESS SCORES (Threshold = 2.50) ---")
scale_mapping = {
    "Very Effective (4)": 4,
    "Effective (3)": 3,
    "Slightly Effective (2)": 2,
    "Not Effective (1)": 1,
    "Not available": np.nan,
}

intervention_cols = [
    col
    for col in df_interv.columns
    if col not in ["Respondent_ID", "Community", "Q36_Support_Needed"]
]

for col in intervention_cols:
  # Map values handling different bracket formats cleanly
  mapped_series = (
      df_interv[col]
      .astype(str)
      .str.extract(r"\((\d)\)")[0]
      .astype(float)
  )
  mean_score = mapped_series.mean()
  status = (
      "Effective / Adequate (>= 2.50)"
      if mean_score >= 2.50
      else "Inadequate (< 2.50)"
  )
  print(f"{col[:50]:50} | Mean: {mean_score:.2f} [{status}]")


--- INTERVENTION EFFECTIVENESS SCORES (Threshold = 2.50) ---
Community drainage maintenance (clearing channels) | Mean: 2.14 [Inadequate (< 2.50)]
Community flood early-warning information          | Mean: 1.93 [Inadequate (< 2.50)]
Assistance from the community to flooed-affected f | Mean: 1.92 [Inadequate (< 2.50)]
Drainage construction and maintenance              | Mean: 1.52 [Inadequate (< 2.50)]
Flood sensitization and awareness campaigns        | Mean: 1.82 [Inadequate (< 2.50)]
Relief materials and emergency response            | Mean: 1.17 [Inadequate (< 2.50)]
State Ministry of Agriculture support (e.g. farm i | Mean: 1.56 [Inadequate (< 2.50)]
Extension advisory services on flood-resistant pra | Mean: 2.30 [Inadequate (< 2.50)]
Flood forecasts and early warnings (e.g. NIHSA Ann | Mean: 2.21 [Inadequate (< 2.50)]
Emergency relief and recovery assistance after flo | Mean: 1.47 [Inadequate (< 2.50)]
